In [ ]:
from pathlib import Path
import os
import sys

def find_project_root(start=None):
    current = Path(start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (
            (candidate / "README.md").exists()
            and (candidate / "src").is_dir()
            and (candidate / "data").is_dir()
            and (candidate / "notebooks").is_dir()
        ):
            return candidate
    raise RuntimeError("Project root was not found. Open this notebook from inside the project folder.")

PROJECT_ROOT = find_project_root()
ROOT = PROJECT_ROOT
root = PROJECT_ROOT
project_root = PROJECT_ROOT

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root configured")
print("Working directory configured")


# Cane Corso Growth Prediction - Linear Regression

This notebook is the first experiment in the project.

The goal is to test whether age in months can be used to predict the weight of a Cane Corso using linear regression.

At this stage, the dataset is a small prototype dataset and is not real veterinary data.


## Problem Statement

Many owners of large-breed dogs want to understand whether their dog is growing in an expected way. This is useful because rapid growth or excessive weight gain can place additional stress on developing bones and joints in large and giant breeds.

In this first experiment, I use a very simple regression setup:

- Input variable: age in months
- Target variable: weight in kilograms

This is only a first stage. More features and better models can be added later.


## Mathematical Formulation

### Input vector `X`

For the regression part, each row is represented as a growth record. In the first experiments, the simplest input is age:

```text
X_simple = [age_months]
```

For richer regression experiments, the input vector can include more growth-related features:

```text
X = [age_months, height_cm, sex_encoded, activity_level_encoded]
```

### Target `y`

The target is the real measured bodyweight:

```text
y = weight_kg
```

### Model function `f(x)`

The model learns a function that estimates bodyweight from the input features:

```text
y_pred = f(X)
```

For simple linear regression:

```text
y_pred = β0 + β1 * age_months
```

For multi-feature regression:

```text
y_pred = β0 + β1x1 + β2x2 + ... + βnxn
```

### Loss function

The main training objective is to reduce prediction error. For ordinary least squares, this is the squared residual loss:

```text
MSE = mean((y_real - y_pred)^2)
```

Regularized models add a penalty term:

```text
Ridge: MSE + λ * sum(βj²)
Lasso: MSE + λ * sum(|βj|)
```

### Metrics

The regression models are evaluated with:

```text
MAE   = mean absolute error
MSE   = mean squared error
RMSE  = square root of MSE
R²    = explained variance score
```

### Interpretation

The model output is interpreted as an expected growth estimate. Residuals show how far a real observation is from the predicted growth pattern:

```text
residual = y_real - y_pred
```

### Limitations

The first regression notebook uses an educational prototype sample, so the results demonstrate the method rather than proving a real veterinary growth standard. The model is sensitive to sample size, data quality, and missing biological factors.


## Mathematical Idea

The first model will use a simple linear regression equation:

$$ y = b_0 + b_1x $$

Where:

- $x$ is the age in months
- $y$ is the predicted weight in kilograms
- $b_0$ is the intercept
- $b_1$ is the coefficient for age


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


## Ordinary Least Squares: Simulated Example

Before using the project data, I create a small simulated example for the Ordinary Least Squares idea.

The goal is to show the lecture concept in a controlled setting:

- one input feature: `age_months`;
- one target: `weight_kg`;
- a mostly linear relationship;
- small measurement noise;
- a fitted line that minimizes squared error.

This example is not used as the final project result. It is included to make the OLS method easier to understand before applying regression to the Cane Corso prototype and the processed public dog growth sample.


In [ ]:
simulated_ols_df = pd.DataFrame({
    "age_months": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
    "weight_kg": [7.5, 10.2, 14.4, 17.1, 20.8, 24.2, 27.5, 31.4, 34.0, 37.2, 40.3, 43.5]
})

X_simulated = simulated_ols_df[["age_months"]]
y_simulated = simulated_ols_df["weight_kg"]

simulated_ols_model = LinearRegression()
simulated_ols_model.fit(X_simulated, y_simulated)

simulated_ols_predictions = simulated_ols_model.predict(X_simulated)

simulated_ols_metrics = pd.DataFrame({
    "Metric": ["Intercept", "Age coefficient", "MAE", "RMSE", "R2"],
    "Value": [
        simulated_ols_model.intercept_,
        simulated_ols_model.coef_[0],
        mean_absolute_error(y_simulated, simulated_ols_predictions),
        mean_squared_error(y_simulated, simulated_ols_predictions) ** 0.5,
        r2_score(y_simulated, simulated_ols_predictions),
    ]
})

simulated_ols_metrics


In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(
    simulated_ols_df["age_months"],
    simulated_ols_df["weight_kg"],
    label="Simulated growth points"
)
plt.plot(
    simulated_ols_df["age_months"],
    simulated_ols_predictions,
    label="OLS fitted line"
)
plt.xlabel("Age in months")
plt.ylabel("Weight in kg")
plt.title("Simulated Ordinary Least Squares Example")
plt.legend()
plt.show()


### Simulated OLS Interpretation

The fitted line represents the expected linear relationship between age and weight in the simulated data.

Ordinary Least Squares chooses the line that minimizes the sum of squared residuals:

```text
residual = real weight - predicted weight
```

In the full project, the same idea is used with actual project features and growth records.


In [ ]:
from pathlib import Path
PROJECT_ROOT = next(candidate for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (candidate / "src").is_dir() and (candidate / "data").is_dir())
if not (PROJECT_ROOT / 'data').exists() and (PROJECT_ROOT.parent / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

data_path = PROJECT_ROOT / 'data' / 'prototype' / 'cane_corso_growth_sample.csv'
df = pd.read_csv(data_path)
df.head()


## Initial Data Exploration

Before training a model, I first check the size, structure, and basic statistics of the dataset.

This helps me understand what data I have and whether the values look usable for a first regression experiment.

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(df["age_months"], df["weight_kg"])
plt.xlabel("Age in months")
plt.ylabel("Weight in kg")
plt.title("Cane Corso Prototype Growth Data: Age vs Weight")
plt.show()

## First Linear Regression Model

In this section, I train the first simple linear regression model.

The model uses age in months as the input feature and weight in kilograms as the target value.

In [ ]:
X = df[["age_months"]]
y = df["weight_kg"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

predictions = pd.DataFrame({
    "actual_weight_kg": y_test.values,
    "predicted_weight_kg": y_pred.round(2)
})

predictions

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

metrics = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R2 Score"],
    "Value": [mae, mse, rmse, r2]
})

metrics

In [ ]:
print("Intercept:", model.intercept_)
print("Age coefficient:", model.coef_[0])

In [ ]:
age_values = pd.DataFrame({
    "age_months": sorted(df["age_months"].unique())
})

predicted_weights = model.predict(age_values)

plt.figure(figsize=(8, 5))
plt.scatter(df["age_months"], df["weight_kg"], label="Actual data")
plt.plot(age_values["age_months"], predicted_weights, label="Linear regression line")
plt.xlabel("Age in months")
plt.ylabel("Weight in kg")
plt.title("Linear Regression: Age vs Weight")
plt.legend()
plt.show()

## First Linear Regression Model

In this section, I train the first simple linear regression model.

The model uses age in months as the input feature and weight in kilograms as the target value.

In [ ]:
X = df[["age_months"]]
y = df["weight_kg"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

predictions = pd.DataFrame({
    "actual_weight_kg": y_test.values,
    "predicted_weight_kg": y_pred.round(2)
})

predictions

In [ ]:
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = mse ** 0.5
r2 = r2_score(y_test, y_pred)

metrics = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R2 Score"],
    "Value": [mae, mse, rmse, r2]
})

metrics

In [ ]:
print("Intercept:", model.intercept_)
print("Age coefficient:", model.coef_[0])

In [ ]:
age_values = pd.DataFrame({
    "age_months": sorted(df["age_months"].unique())
})

predicted_weights = model.predict(age_values)

plt.figure(figsize=(8, 5))
plt.scatter(df["age_months"], df["weight_kg"], label="Actual data")
plt.plot(age_values["age_months"], predicted_weights, label="Linear regression line")
plt.xlabel("Age in months")
plt.ylabel("Weight in kg")
plt.title("Linear Regression: Age vs Weight")
plt.legend()
plt.show()

## Result Interpretation

The first linear regression model shows how weight changes with age in the prototype dataset.

The model uses only one feature: age in months. Because of this, it is easy to understand, but it is also limited.

The evaluation metrics help me check how close the predictions are to the actual values.

- MAE shows the average prediction error in kilograms.
- RMSE gives more weight to larger errors.
- R2 Score shows how much of the variation in weight is explained by age.

This first experiment is useful as a baseline model. However, real dog growth depends on more factors than age alone, such as sex, genetics, nutrition, activity level, health, and environment.

Because of that, this model should not be used as a veterinary tool. It is only a first educational machine learning experiment.

## Polynomial Regression

The previous model used a straight line to predict weight from age.

Dog growth is usually not perfectly linear. Puppies often grow faster in the first months and then the growth speed becomes slower.

For this reason, I will test a polynomial regression model as an extension of simple linear regression.

In [ ]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

In [ ]:
X = df[["age_months"]]
y = df["weight_kg"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

poly_model = Pipeline([
    ("polynomial_features", PolynomialFeatures(degree=2)),
    ("linear_regression", LinearRegression())
])

poly_model.fit(X_train, y_train)

poly_y_pred = poly_model.predict(X_test)

poly_predictions = pd.DataFrame({
    "actual_weight_kg": y_test.values,
    "predicted_weight_kg": poly_y_pred.round(2)
})

poly_predictions

In [ ]:
poly_mae = mean_absolute_error(y_test, poly_y_pred)
poly_mse = mean_squared_error(y_test, poly_y_pred)
poly_rmse = poly_mse ** 0.5
poly_r2 = r2_score(y_test, poly_y_pred)

poly_metrics = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R2 Score"],
    "Polynomial Regression": [poly_mae, poly_mse, poly_rmse, poly_r2]
})

poly_metrics

In [ ]:
age_range = pd.DataFrame({
    "age_months": sorted(df["age_months"].unique())
})

poly_predicted_weights = poly_model.predict(age_range)

plt.figure(figsize=(8, 5))
plt.scatter(df["age_months"], df["weight_kg"], label="Actual data")
plt.plot(age_range["age_months"], poly_predicted_weights, label="Polynomial regression line")
plt.xlabel("Age in months")
plt.ylabel("Weight in kg")
plt.title("Polynomial Regression: Age vs Weight")
plt.legend()
plt.show()

### Polynomial Regression Interpretation

The polynomial regression model allows the prediction curve to bend instead of using only a straight line.

This can be useful for growth data because growth is often faster at early ages and slower later.

However, this is still a simple model because it uses only age in months as the input feature. More realistic models should also include sex, height, activity level, nutrition, and other factors.

## Multi-Dimensional Linear Regression

The previous models used only age in months as the input feature.

In a real growth analysis problem, weight can depend on more than one factor. For this reason, I will test a multi-dimensional linear regression model.

This model will use:

- age in months
- height in centimeters
- sex
- activity level

Categorical values such as sex and activity level must be converted into numeric values before training the model.

In [ ]:
multi_features = pd.get_dummies(
    df[["age_months", "height_cm", "sex", "activity_level"]],
    drop_first=True
)

multi_target = df["weight_kg"]

multi_features.head()

In [ ]:
X_train_multi, X_test_multi, y_train_multi, y_test_multi = train_test_split(
    multi_features,
    multi_target,
    test_size=0.25,
    random_state=42
)

multi_model = LinearRegression()
multi_model.fit(X_train_multi, y_train_multi)

multi_y_pred = multi_model.predict(X_test_multi)

multi_predictions = pd.DataFrame({
    "actual_weight_kg": y_test_multi.values,
    "predicted_weight_kg": multi_y_pred.round(2)
})

multi_predictions

In [ ]:
multi_mae = mean_absolute_error(y_test_multi, multi_y_pred)
multi_mse = mean_squared_error(y_test_multi, multi_y_pred)
multi_rmse = multi_mse ** 0.5
multi_r2 = r2_score(y_test_multi, multi_y_pred)

multi_metrics = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R2 Score"],
    "Multi-Dimensional Linear Regression": [multi_mae, multi_mse, multi_rmse, multi_r2]
})

multi_metrics

In [ ]:
multi_coefficients = pd.DataFrame({
    "Feature": multi_features.columns,
    "Coefficient": multi_model.coef_
})

multi_coefficients

### Multi-Dimensional Regression Interpretation

The multi-dimensional linear regression model uses more information than the first simple linear regression model.

This makes the experiment more realistic because dog weight can depend on age, height, sex, and activity level.

However, the dataset is still small and only a prototype. The results should be interpreted carefully. This model is useful for learning how multiple input features can be used in regression, but it is not enough for real veterinary or health-related decisions.

## Regularization: Ridge and Lasso Regression

Linear regression can sometimes become too sensitive to the training data, especially when more features are added.

Regularization is used to control the model coefficients and reduce overfitting.

In this section, I will test two regularized regression models:

- Ridge Regression
- Lasso Regression

Both models will use the same multi-dimensional features from the previous experiment.

In [ ]:
from sklearn.linear_model import Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [ ]:
regularized_features = pd.get_dummies(
    df[["age_months", "height_cm", "sex", "activity_level"]],
    drop_first=True
)

regularized_target = df["weight_kg"]

X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    regularized_features,
    regularized_target,
    test_size=0.25,
    random_state=42
)

regularized_features.head()

In [ ]:
ridge_model = Pipeline([
    ("scaler", StandardScaler()),
    ("ridge", Ridge(alpha=1.0))
])

ridge_model.fit(X_train_reg, y_train_reg)

ridge_y_pred = ridge_model.predict(X_test_reg)

ridge_mae = mean_absolute_error(y_test_reg, ridge_y_pred)
ridge_mse = mean_squared_error(y_test_reg, ridge_y_pred)
ridge_rmse = ridge_mse ** 0.5
ridge_r2 = r2_score(y_test_reg, ridge_y_pred)

ridge_results = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R2 Score"],
    "Ridge Regression": [ridge_mae, ridge_mse, ridge_rmse, ridge_r2]
})

ridge_results

In [ ]:
lasso_model = Pipeline([
    ("scaler", StandardScaler()),
    ("lasso", Lasso(alpha=0.1, max_iter=10000))
])

lasso_model.fit(X_train_reg, y_train_reg)

lasso_y_pred = lasso_model.predict(X_test_reg)

lasso_mae = mean_absolute_error(y_test_reg, lasso_y_pred)
lasso_mse = mean_squared_error(y_test_reg, lasso_y_pred)
lasso_rmse = lasso_mse ** 0.5
lasso_r2 = r2_score(y_test_reg, lasso_y_pred)

lasso_results = pd.DataFrame({
    "Metric": ["MAE", "MSE", "RMSE", "R2 Score"],
    "Lasso Regression": [lasso_mae, lasso_mse, lasso_rmse, lasso_r2]
})

lasso_results

In [ ]:
regularization_comparison = pd.DataFrame({
    "Model": ["Ridge Regression", "Lasso Regression"],
    "MAE": [ridge_mae, lasso_mae],
    "MSE": [ridge_mse, lasso_mse],
    "RMSE": [ridge_rmse, lasso_rmse],
    "R2 Score": [ridge_r2, lasso_r2]
})

regularization_comparison

### Regularization Interpretation

Ridge and Lasso Regression are regularized versions of linear regression.

Ridge reduces the size of the coefficients but usually keeps all features in the model.

Lasso can reduce some coefficients strongly and may even set some of them close to zero.

In this prototype project, regularization is useful because the model uses multiple input features. However, the dataset is still small, so the results should be interpreted only as an educational experiment.

## RANSAC Robust Regression

RANSAC is a robust regression method that can reduce the influence of outliers.

This is useful because real-world measurements may contain errors. For example, a dog weight value may be entered incorrectly by the owner.

In this experiment, I add one artificial outlier to the prototype dataset and compare a normal linear regression model with a RANSAC regression model.

In [ ]:
from sklearn.linear_model import RANSACRegressor

In [ ]:
outlier = pd.DataFrame({
    "dog_id": [99],
    "dog_name": ["Outlier"],
    "sex": ["male"],
    "age_months": [6],
    "weight_kg": [65.0],
    "height_cm": [56],
    "activity_level": ["medium"],
    "source_type": ["artificial_outlier"]
})

ransac_df = pd.concat([df, outlier], ignore_index=True)

ransac_df.tail()

In [ ]:
X_ransac = ransac_df[["age_months"]]
y_ransac = ransac_df["weight_kg"]

normal_model_with_outlier = LinearRegression()
normal_model_with_outlier.fit(X_ransac, y_ransac)

ransac_model = RANSACRegressor(
    estimator=LinearRegression(),
    residual_threshold=5.0,
    random_state=42
)

ransac_model.fit(X_ransac, y_ransac)

age_range_ransac = pd.DataFrame({
    "age_months": sorted(ransac_df["age_months"].unique())
})

normal_outlier_predictions = normal_model_with_outlier.predict(age_range_ransac)
ransac_predictions = ransac_model.predict(age_range_ransac)

In [ ]:
plt.figure(figsize=(8, 5))

plt.scatter(
    ransac_df["age_months"],
    ransac_df["weight_kg"],
    label="Data with artificial outlier"
)

plt.plot(
    age_range_ransac["age_months"],
    normal_outlier_predictions,
    label="Linear regression with outlier"
)

plt.plot(
    age_range_ransac["age_months"],
    ransac_predictions,
    label="RANSAC regression"
)

plt.xlabel("Age in months")
plt.ylabel("Weight in kg")
plt.title("RANSAC Regression Compared to Linear Regression")
plt.legend()
plt.show()

### RANSAC Interpretation

The artificial outlier represents a possible measurement or data entry error.

A normal linear regression model can be influenced by this outlier because it tries to fit all data points.

RANSAC tries to focus on the main pattern in the data and reduce the influence of abnormal points.

This makes RANSAC useful for real-world datasets where some values may be incorrect or unusual.

## Linear Regression on Processed Public Dog Growth Data

The first regression experiments use the small Cane Corso prototype dataset. To connect the notebook more directly to real data, this section also trains regression models on the processed public dog growth sample.

The public dataset is broader than Cane Corso. In this project, it is used as a real-world growth-data foundation while the Cane Corso domain remains the product story and target use case.

This section supports the lecture requirement for implementation on real data.


In [ ]:
real_data_path = PROJECT_ROOT / 'data' / 'processed' / 'dog_growth_public_sample.csv'
real_growth_df = pd.read_csv(real_data_path)

real_growth_df = real_growth_df.dropna(
    subset=["visit_age_months", "weight_kg"]
).copy()

# The project focuses on growth monitoring, so I keep puppy and young-dog records for this example.
real_growth_df = real_growth_df[
    (real_growth_df["visit_age_months"] >= 0)
    & (real_growth_df["visit_age_months"] <= 36)
].copy()

real_growth_df[["visit_age_months", "weight_kg", "average_adult_breed_weight_kg"]].describe()


In [ ]:
X_real_simple = real_growth_df[["visit_age_months"]]
y_real = real_growth_df["weight_kg"]

X_train_real, X_test_real, y_train_real, y_test_real = train_test_split(
    X_real_simple,
    y_real,
    test_size=0.2,
    random_state=42
)

real_simple_model = LinearRegression()
real_simple_model.fit(X_train_real, y_train_real)

real_simple_predictions = real_simple_model.predict(X_test_real)

real_simple_metrics = pd.DataFrame({
    "Model": ["Real Data Simple Linear Regression"],
    "MAE": [mean_absolute_error(y_test_real, real_simple_predictions)],
    "MSE": [mean_squared_error(y_test_real, real_simple_predictions)],
    "RMSE": [mean_squared_error(y_test_real, real_simple_predictions) ** 0.5],
    "R2": [r2_score(y_test_real, real_simple_predictions)]
})

real_simple_metrics


In [ ]:
real_multi_df = real_growth_df.dropna(
    subset=["visit_age_months", "weight_kg", "average_adult_breed_weight_kg", "gender"]
).copy()

real_multi_features = pd.get_dummies(
    real_multi_df[["visit_age_months", "average_adult_breed_weight_kg", "gender"]],
    drop_first=True
)
real_multi_target = real_multi_df["weight_kg"]

X_train_real_multi, X_test_real_multi, y_train_real_multi, y_test_real_multi = train_test_split(
    real_multi_features,
    real_multi_target,
    test_size=0.2,
    random_state=42
)

real_multi_model = LinearRegression()
real_multi_model.fit(X_train_real_multi, y_train_real_multi)

real_multi_predictions = real_multi_model.predict(X_test_real_multi)

real_multi_metrics = pd.DataFrame({
    "Model": ["Real Data Multi-Feature Linear Regression"],
    "MAE": [mean_absolute_error(y_test_real_multi, real_multi_predictions)],
    "MSE": [mean_squared_error(y_test_real_multi, real_multi_predictions)],
    "RMSE": [mean_squared_error(y_test_real_multi, real_multi_predictions) ** 0.5],
    "R2": [r2_score(y_test_real_multi, real_multi_predictions)]
})

real_multi_metrics


In [ ]:
real_data_regression_comparison = pd.concat(
    [real_simple_metrics, real_multi_metrics],
    ignore_index=True
)

real_data_regression_comparison.sort_values(by="RMSE")


### Real Data Regression Interpretation

The real-data regression experiment is intentionally separate from the Cane Corso prototype experiment.

The prototype sample makes the domain idea easy to understand. The processed public sample shows the same regression workflow on a larger real dataset.

The multi-feature version can use more context than age alone, for example the average adult breed weight and sex/gender information. This usually gives the model a better representation of expected bodyweight than a single-feature line.


## Final Model Comparison

In this section, I compare the regression models tested in this notebook.

The comparison helps me understand which model performs better on the current prototype dataset.

The models are compared using:

- MAE
- RMSE
- R2 Score

Lower MAE and RMSE are better. Higher R2 Score is better.

In [ ]:
final_model_comparison = pd.DataFrame({
    "Model": [
        "Simple Linear Regression",
        "Polynomial Regression",
        "Multi-Dimensional Linear Regression",
        "Ridge Regression",
        "Lasso Regression"
    ],
    "Main Features": [
        "age_months",
        "age_months with polynomial degree 2",
        "age_months, height_cm, sex, activity_level",
        "age_months, height_cm, sex, activity_level",
        "age_months, height_cm, sex, activity_level"
    ],
    "MAE": [
        mae,
        poly_mae,
        multi_mae,
        ridge_mae,
        lasso_mae
    ],
    "RMSE": [
        rmse,
        poly_rmse,
        multi_rmse,
        ridge_rmse,
        lasso_rmse
    ],
    "R2 Score": [
        r2,
        poly_r2,
        multi_r2,
        ridge_r2,
        lasso_r2
    ]
})

final_model_comparison

In [ ]:
final_model_comparison.sort_values(by="RMSE")

## Final Notes for This Topic

The first simple linear regression model is easy to understand and works as a baseline.

Polynomial regression is useful because dog growth is not always perfectly linear.

Multi-dimensional regression is more realistic because it uses more than one feature.

Ridge and Lasso add regularization, which can help control model coefficients when more features are used.

RANSAC is useful when the dataset may contain outliers or incorrect measurements.

At this stage, the dataset is still a small prototype dataset, so the results should not be interpreted as real veterinary conclusions. The main goal is to apply the course topic gradually and understand how different regression methods behave.